# MediFlow 기존 데이터 최종 Test 비교 · 실행 수정본

GPU 런타임을 선택하고 **모두 실행**한 뒤 Drive를 연결합니다. 이미 Drive에 있는 공통 ZIP v3,
세 원본 데이터 ZIP, 1·2·3차 모델을 읽습니다. 실제 촬영 사진의 복호화 키는 사용하지 않습니다.
별도 실행 환경의 그래프 출력 설정을 자동 처리합니다. 이 노트북을 열고 모두 실행하면 됩니다.

피부 700장, 얼굴 400장, 두피 1,252장의 원본 Test를 확인하고 학습 전·1차·2차·3차 모델을
같은 원본·조명 변환·확장 변환으로 평가합니다. 분야별 후보는 Validation으로 고정했습니다.
Test 점수로 모델을 다시 고르지 않습니다. 새 학습은 수행하지 않습니다.

결과는 Drive `MediFlow_adaptation_runs/4ae6e07eb4af1225/final_test_bc4d235c1354`에 저장합니다.
CSV에는 정확도·Macro-F1·학습 전 대비 변화, 분야별 JSON에는 클래스별 지표와 혼동행렬이 있습니다.
클래스별 CSV와 비교 그래프도 저장하며, 완료되면 결과 ZIP 다운로드를 시작합니다.
중단되면 같은 노트북에서 이미 완료된 분야 결과를 재사용합니다.
Python 환경 준비부터 예측까지 실제 출력과 오류를 화면에 표시합니다.
실행 로그는 결과 폴더의 `logs`에 남습니다. 실패하면 마지막 실제 오류 문구를 확인하세요.


In [ ]:
import hashlib
import os
import shutil
import signal
import subprocess
import sys
import tempfile
import uuid
from pathlib import Path, PurePosixPath
from zipfile import ZipFile

from google.colab import drive

print("MediFlow 최종 Test · 그래프 출력 설정 자동 처리 수정본", flush=True)
drive.mount("/content/drive")
my_drive = Path("/content/drive/MyDrive")
project = my_drive / "mediflow_Project"
bundle = project / "MediFlow_Adaptation_Common_v3.zip"
run_root = my_drive / "MediFlow_adaptation_runs" / "4ae6e07eb4af1225"
stages = [run_root / f"stage{n}" for n in (1, 2, 3)]
out = run_root / "final_test_bc4d235c1354"
if not all((stage / "bundle.json").is_file() for stage in stages):
    raise FileNotFoundError("Drive에서 1·2·3차 완료 폴더를 확인하세요.")
log_root = out / "logs"
log_root.mkdir(parents=True, exist_ok=True)
run_id = uuid.uuid4().hex[:12]
log_path = log_root / f"run_{run_id}.log"
print("실행 로그 저장 위치", log_path, flush=True)

with tempfile.TemporaryDirectory(prefix="mediflow_final_test_", dir="/content") as temp:
    work = Path(temp)
    runtime = work / "runtime"
    local_bundle = work / bundle.name
    print("공통 ZIP v3 확인 중", flush=True)
    shutil.copyfile(bundle, local_bundle)
    with local_bundle.open("rb") as handle:
        if hashlib.file_digest(handle, "sha256").hexdigest() != "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe":
            raise ValueError("공통 ZIP v3의 해시가 일치하지 않습니다.")
    with ZipFile(local_bundle) as archive:
        for member in archive.infolist():
            path = PurePosixPath(member.filename)
            if member.is_dir() or not path.parts or path.parts[0] not in (
                "src", "configs", "references", "results"
            ):
                continue
            if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
                raise ValueError("공통 ZIP에 예상치 못한 경로가 있습니다.")
            archive.extract(member, runtime)
    local_bundle.unlink()
    for name, contents in {'adaptation_final_test.py': b'"""Run the locked camera-adaptation comparison on the original labeled Test split."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport io\nimport platform\nimport tempfile\nimport zipfile\nfrom contextlib import nullcontext\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport cv2\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import adaptation_stage1 as training\nfrom mediflow_datasets import camera_extended as extended\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as experiment\nfrom mediflow_datasets.adaptation_stage1_colab import copy_source\nfrom mediflow_datasets.domain_adaptation_batch import digest, find_source\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nCONDITIONS = ("clean", "lighting", "expanded")\nMODEL_LABELS = {\n    "initial": "Before adaptation",\n    "stage1_lighting": "Stage 1",\n    "stage2_expanded": "Stage 2",\n    "stage3_aligned": "Stage 3",\n}\nDOMAIN_LABELS = {"skin": "Microscope skin", "web_skin": "Webcam face", "hair": "Microscope scalp"}\n\n\ndef prepare_test(archive, task, lock, manifest):\n    if digest(manifest) != lock["manifest_sha256"]:\n        raise ValueError(f"{task[\'domain\']}: Test manifest SHA-256 mismatch")\n    rows, _ = data.source_index(archive, task["classes"], task["source_sha256"])\n    original_test = rows["original", "test"]\n    declared = data.read_json(manifest)["original"]["test"]\n    actual = [(row.path, row.label) for row in original_test]\n    expected = [(row["path"], row["label"]) for row in declared]\n    if actual != expected or len(actual) != lock["test_count"]:\n        raise ValueError(f"{task[\'domain\']}: original Test rows differ from the locked manifest")\n    return original_test\n\n\ndef evaluate_zip(archive_path, rows, task, spec, reference, predictors, seed):\n    """Read each Test image once; feed the exact same arrays to all model arms."""\n    scores = {name: {condition: [] for condition in CONDITIONS} for name in predictors}\n    size, batch_size = spec["size"], task["batch_size"]\n    with zipfile.ZipFile(archive_path) as archive:\n        for start in range(0, len(rows), batch_size):\n            batch = rows[start : start + batch_size]\n            clean, lighting, expanded = [], [], []\n            for row in batch:\n                image = experiment.image_tensor(archive.read(row.path), size, task["input_policy"])\n                item_seed = experiment.item_seed(row, seed)\n                clean.append(image)\n                lighting.append(\n                    experiment.transform_image(\n                        image, reference["extended_profile"]["lighting"], "target_guided", item_seed\n                    )\n                )\n                expanded.append(\n                    extended.transform(image, reference["extended_profile"], "expanded", item_seed)\n                )\n            for condition, inputs in zip(CONDITIONS, (clean, lighting, expanded), strict=True):\n                tensor = tf.convert_to_tensor(np.stack(inputs), dtype=tf.float32)\n                for name, predict in predictors.items():\n                    result = predict(tensor).numpy()\n                    if result.shape != (len(batch), len(task["classes"])):\n                        raise ValueError(f"{name}: unexpected model output shape {result.shape}")\n                    scores[name][condition].append(result)\n            done = start + len(batch)\n            if start == 0 or start % (batch_size * 20) == 0 or done == len(rows):\n                training.show(\n                    f"[{task[\'domain\']}] Test \xec\x98\x88\xec\xb8\xa1 \xc2\xb7 \xeb\xaa\xa8\xeb\x8d\xb8 4\xea\xb0\x9c \xc3\x97 \xec\xa1\xb0\xea\xb1\xb4 3\xea\xb0\x9c", done, len(rows)\n                )\n    return {\n        name: {\n            condition: experiment.classification(\n                rows, np.concatenate(batches), len(task["classes"])\n            )\n            for condition, batches in results.items()\n        }\n        for name, results in scores.items()\n    }\n\n\ndef domain_result(runtime, stages, domain, task, lock, seed, archive):\n    print(f"[{domain}] \xec\x9b\x90\xeb\xb3\xb8 ZIP \xed\x95\xb4\xec\x8b\x9c\xec\x99\x80 Test \xeb\xaa\xa9\xeb\xa1\x9d \xed\x99\x95\xec\x9d\xb8", flush=True)\n    rows = prepare_test(archive, task, lock, stages[0] / domain / "source_manifest.json")\n    reference = data.read_json(runtime / "references" / f"{domain}.json")\n    if training.object_hash(reference) != lock["reference_sha256"]:\n        raise ValueError(f"{domain}: camera profile mismatch")\n    spec = task["models"][lock["start_role"]]\n    predictors = {}\n    for name, model in lock["models"].items():\n        root = runtime if model["stage"] == 0 else stages[model["stage"] - 1]\n        path = data.local_path(root, model["path"])\n        if digest(path) != model["sha256"]:\n            raise ValueError(f"{domain}/{name}: checkpoint SHA-256 mismatch")\n        print(f"[{domain}] {name} \xeb\xaa\xa8\xeb\x8d\xb8 \xeb\xa1\x9c\xeb\x94\xa9", flush=True)\n        predictors[name] = training.predictor(keras.models.load_model(path, compile=False), spec)\n    metrics = evaluate_zip(archive, rows, task, spec, reference, predictors, seed)\n    return {\n        "domain": domain,\n        "classes": task["classes"],\n        "test_count": len(rows),\n        "source_sha256": task["source_sha256"],\n        "reference_sha256": lock["reference_sha256"],\n        "input_policy": task["input_policy"],\n        "selected_on_validation": lock["selected_on_validation"],\n        "metrics": metrics,\n    }\n\n\ndef write_comparison(output, results):\n    stream = io.StringIO()\n    writer = csv.writer(stream)\n    writer.writerow(\n        [\n            "domain",\n            "model",\n            "selected_on_validation",\n            "condition",\n            "count",\n            "accuracy",\n            "macro_f1",\n            "delta_accuracy_vs_initial",\n            "delta_macro_f1_vs_initial",\n        ]\n    )\n    for result in results:\n        for model, conditions in result["metrics"].items():\n            for condition, metrics in conditions.items():\n                baseline = result["metrics"]["initial"][condition]\n                writer.writerow(\n                    [\n                        result["domain"],\n                        model,\n                        model == result["selected_on_validation"],\n                        condition,\n                        metrics["labeled_count"],\n                        metrics["accuracy"],\n                        metrics["macro_f1"],\n                        metrics["accuracy"] - baseline["accuracy"],\n                        metrics["macro_f1"] - baseline["macro_f1"],\n                    ]\n                )\n    (output / "test_comparison.csv").write_text(stream.getvalue(), encoding="utf-8", newline="")\n\n\ndef write_class_metrics(output, results):\n    with (output / "class_metrics.csv").open("w", encoding="utf-8-sig", newline="") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(\n            [\n                "domain",\n                "model",\n                "condition",\n                "class_index",\n                "class_name",\n                "support",\n                "precision",\n                "recall",\n                "f1",\n            ]\n        )\n        for result in results:\n            for model, conditions in result["metrics"].items():\n                for condition, metrics in conditions.items():\n                    for index, name in enumerate(result["classes"]):\n                        writer.writerow(\n                            [\n                                result["domain"],\n                                model,\n                                condition,\n                                index,\n                                name,\n                                metrics["support"][index],\n                                metrics["precision"][index],\n                                metrics["recall"][index],\n                                metrics["class_f1"][index],\n                            ]\n                        )\n\n\ndef write_plots(output, results):\n    import matplotlib\n\n    matplotlib.use("Agg")\n    from matplotlib import pyplot as plt\n\n    plots = output / "plots"\n    plots.mkdir(exist_ok=True)\n    fig, axes = plt.subplots(3, 2, figsize=(13, 11), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            for column, metric in enumerate(("accuracy", "macro_f1")):\n                axis = axes[row, column]\n                for model, conditions in result["metrics"].items():\n                    label = MODEL_LABELS[model]\n                    if model == result["selected_on_validation"]:\n                        label += " (fixed on Validation)"\n                    axis.plot(\n                        range(3),\n                        [conditions[c][metric] for c in CONDITIONS],\n                        marker="o",\n                        label=label,\n                    )\n                axis.set_xticks(range(3), ["Original", "Lighting", "Expanded"])\n                axis.set_ylim(0, 1.01)\n                axis.set_title(f"{DOMAIN_LABELS[result[\'domain\']]} \xc2\xb7 {metric}")\n                axis.grid(alpha=0.2)\n                axis.legend(fontsize=8)\n        fig.suptitle("MediFlow \xc2\xb7 Original labeled Test comparison", fontsize=16)\n        fig.savefig(plots / "test_comparison.png", dpi=180)\n    finally:\n        plt.close(fig)\n    fig, axes = plt.subplots(3, 3, figsize=(16, 14), layout="constrained")\n    try:\n        for row, result in enumerate(results):\n            model = result["selected_on_validation"]\n            for column, condition in enumerate(CONDITIONS):\n                axis = axes[row, column]\n                matrix = np.asarray(result["metrics"][model][condition]["confusion_matrix"])\n                axis.imshow(matrix, cmap="Blues", vmin=0)\n                indexes = range(len(result["classes"]))\n                axis.set_xticks(indexes)\n                axis.set_yticks(indexes)\n                axis.set_xlabel("Predicted class index")\n                axis.set_ylabel("True class index")\n                axis.set_title(\n                    f"{DOMAIN_LABELS[result[\'domain\']]} \xc2\xb7 {MODEL_LABELS[model]} \xc2\xb7 {condition}"\n                )\n                for i, j in np.ndindex(matrix.shape):\n                    axis.text(\n                        j,\n                        i,\n                        str(matrix[i, j]),\n                        ha="center",\n                        va="center",\n                        fontsize=7,\n                        color="white" if matrix[i, j] > matrix.max() / 2 else "black",\n                    )\n        fig.suptitle("Validation-fixed candidates \xc2\xb7 Class names in class_metrics.csv", fontsize=16)\n        fig.savefig(plots / "selected_confusion_matrices.png", dpi=180)\n    finally:\n        plt.close(fig)\n\n\ndef run(runtime, stages, dataset_root, plan_path, output, work=None):\n    plan = data.read_json(plan_path)\n    config = data.read_json(runtime / "configs" / "adaptation_stage1.json")\n    if training.object_hash(config) != plan["configuration_sha256"]:\n        raise ValueError("Common bundle configuration differs from the locked plan")\n    if digest(training.__file__) != plan["predictor_code_sha256"]:\n        raise ValueError("Predictor differs from the Stage 3 validation code")\n    identity = {"plan_sha256": digest(plan_path), "evaluator_sha256": digest(__file__)}\n    output.mkdir(parents=True, exist_ok=True)\n    marker = output / "evaluation_plan.json"\n    record = {**identity, "plan": plan}\n    if marker.exists() and data.read_json(marker) != record:\n        raise ValueError("Output belongs to another evaluation plan")\n    write_progress(marker, record)\n    code = output / "code"\n    code.mkdir(exist_ok=True)\n    for path in (Path(__file__), Path(training.__file__)):\n        (code / path.name).write_bytes(path.read_bytes())\n    results = []\n    for domain in training.DOMAINS:\n        task, lock = config["tasks"][domain], plan["domains"][domain]\n        destination = output / f"{domain}.json"\n        if destination.is_file():\n            result = data.read_json(destination)\n            if result["identity"] != identity:\n                raise ValueError(f"{domain}: saved result identity mismatch")\n            print(f"[{domain}] \xeb\x8f\x99\xec\x9d\xbc \xea\xb3\x84\xed\x9a\x8d\xec\x9d\x98 \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        else:\n            source = find_source(dataset_root, task)\n            with tempfile.TemporaryDirectory(dir=work) if work else nullcontext() as scratch:\n                archive = source\n                if scratch:\n                    archive = Path(scratch) / "source.zip"\n                    copy_source(source, archive)\n                result = domain_result(\n                    runtime, stages, domain, task, lock, plan["transform_seed"], archive\n                )\n            result.update(\n                identity=identity,\n                environment={\n                    "python": platform.python_version(),\n                    "tensorflow": tf.__version__,\n                    "keras": keras.__version__,\n                    "numpy": np.__version__,\n                    "opencv": cv2.__version__,\n                    "matplotlib": version("matplotlib"),\n                    "gpu": [\n                        tf.config.experimental.get_device_details(d)\n                        for d in tf.config.list_physical_devices("GPU")\n                    ],\n                },\n            )\n            write_progress(destination, result)\n            keras.backend.clear_session()\n        results.append(result)\n    write_comparison(output, results)\n    write_class_metrics(output, results)\n    write_plots(output, results)\n    write_progress(\n        output / "summary.json",\n        {\n            **identity,\n            "completed_domains": [r["domain"] for r in results],\n            "comparisons": 36,\n            "source_test_evaluated": True,\n            "real_device_accuracy": None,\n        },\n    )\n    print(f"\xec\xb5\x9c\xec\xa2\x85 Test \xea\xb2\xb0\xea\xb3\xbc: {output / \'test_comparison.csv\'}", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--runtime", type=Path, required=True)\n    parser.add_argument("--stages", type=Path, nargs=3, required=True)\n    parser.add_argument("--datasets", type=Path, required=True)\n    parser.add_argument("--plan", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--work", type=Path)\n    args = parser.parse_args()\n    run(args.runtime, args.stages, args.datasets, args.plan, args.output, args.work)\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1.py': b'"""Compare two eligible starting models per domain with the same lighting treatment."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport math\nimport os\nimport shutil\nimport time\nimport uuid\nimport zipfile\nfrom collections import Counter\nfrom concurrent.futures import ThreadPoolExecutor\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport keras\nimport numpy as np\nimport tensorflow as tf\n\nfrom mediflow_datasets import camera_extended, domain_alignment\nfrom mediflow_datasets import domain_adaptation_data as data\nfrom mediflow_datasets import domain_adaptation_experiment as exp\nfrom mediflow_datasets import domain_adaptation_resolution as resolution\nfrom mediflow_datasets.candidate_reproduction import ROOT\nfrom mediflow_datasets.domain_adaptation_io import write_progress\nfrom mediflow_datasets.domain_adaptation_training import object_hash\n\nDOMAINS = ("skin", "web_skin", "hair")\nARMS = ("basic", "selected")\nWORKERS = min(4, os.cpu_count() or 1)\n\n\ndef show(label, step, total, extra=""):\n    print(f"{label} | {step}/{total} ({100 * step / total:.1f}%) {extra}", flush=True)\n\n\ndef prepare_source(domain, archive_path, task, reference, out):\n    classes = task["classes"]\n    print(f"[{domain}] \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP \xed\x95\xb4\xec\x8b\x9c\xc2\xb7\xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4 \xed\x99\x95\xec\x9d\xb8", flush=True)\n    source, sha = data.source_index(archive_path, classes, task["source_sha256"])\n    retained = (\n        resolution.filtered_source(source, reference["quarantine"], classes, sha)\n        if reference.get("quarantine")\n        else {k: v for k, v in source.items() if k[0] == "original"}\n    )\n    # Skin\'s historical distinction is Original versus Augmented preparation. Use only\n    # linked descendants of retained originals. Adaptation itself uses identical originals.\n    train = retained["original", "train"]\n    originals = {r.path: r for r in train}\n    prepare_augmented = domain == "skin" and any(\n        spec.get("skin_augmented_preparation") for spec in task.get("models", {}).values()\n    )\n    descendants = [\n        r\n        for r in source["augmented", "train"]\n        if prepare_augmented\n        and r.origin in originals\n        and r.label == originals[r.origin].label\n        and r.path.rsplit("/", 1)[-1] != r.origin.rsplit("/", 1)[-1]\n    ]\n    # For the saved Hair model, include its previous augmented source in the leakage scan.\n    scanned = (\n        [r for rows in source.values() for r in rows]\n        if domain == "hair"\n        else [r for rows in retained.values() for r in rows] + descendants\n    )\n    label = f"[{domain}] \xec\xa4\x91\xeb\xb3\xb5\xc2\xb7\xeb\xb6\x84\xed\x95\xa0 \xed\x99\x95\xec\x9d\xb8 (\xec\x82\xac\xec\xa7\x84)"\n    entries = []\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def signature(row):\n            return row, *data.image_signatures(archive.read(row.path))\n\n        for start in range(0, len(scanned), 128):\n            entries.extend(pool.map(signature, scanned[start : start + 128]))\n            done = len(entries)\n            if start == 0 or done % 1024 == 0 or done == len(scanned):\n                show(label, done, len(scanned))\n    report = data.audit_groups(entries)\n    dropped = set()\n    if descendants:\n        involved = {p for pair in report["near_duplicate_review"] for p in pair}\n        involved.update(\n            p for leak in report["blocking_leaks_or_conflicts"] for p in leak.get("paths", ())\n        )\n        dropped = {r.path for r in descendants if r.path in involved}\n        entries = [entry for entry in entries if entry[0].path not in dropped]\n        report = data.audit_groups(entries)\n    report["excluded_augmented_descendants"] = sorted(dropped)\n    report["source_target_pixel_overlap"] = len(\n        {entry[2] for entry in entries} & set(reference["pixel_hashes"])\n    )\n    report["source_sha256"] = sha\n    report["retained_counts"] = {split: len(retained["original", split]) for split in data.SPLITS}\n    if domain == "hair":\n        consistency = {}\n        for split in data.SPLITS:\n            original, augmented = (\n                Counter(\n                    (row.label, pixel)\n                    for row, _, pixel, _ in entries\n                    if row.split == split and row.variant == variant\n                )\n                for variant in ("original", "augmented")\n            )\n            consistency[split] = (\n                not (original - augmented) if split == "train" else original == augmented\n            )\n        report["original_augmented_consistency"] = consistency\n    write_progress(out / "data_audit.json", report)\n    if (\n        report["blocking_leaks_or_conflicts"]\n        or report["near_duplicate_review"]\n        or report["source_target_pixel_overlap"]\n        or not all(report.get("original_augmented_consistency", {}).values())\n    ):\n        raise ValueError(f"{domain} \xeb\xb6\x84\xed\x95\xa0 \xec\xa0\x90\xea\xb2\x80 \xec\x8b\xa4\xed\x8c\xa8. data_audit.json\xec\x9d\x84 \xed\x99\x95\xec\x9d\xb8\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    pixel_by_path = {row.path: pixel for row, _, pixel, _ in entries}\n    paired = {}\n    for row in descendants:\n        if row.path not in dropped and pixel_by_path[row.path] != pixel_by_path[row.origin]:\n            paired.setdefault(row.origin, row)\n    if prepare_augmented and {r.label for r in paired.values()} != set(range(len(classes))):\n        raise ValueError("\xed\x94\xbc\xeb\xb6\x80 \xec\x84\xa0\xec\xa0\x95 \xeb\xb0\xa9\xec\x8b\x9d \xec\x9e\xac\xed\x98\x84\xec\x97\x90 \xed\x95\x84\xec\x9a\x94\xed\x95\x9c \xed\x81\xb4\xeb\x9e\x98\xec\x8a\xa4\xeb\xb3\x84 \xed\x8c\x8c\xec\x83\x9d\xeb\xb3\xb8\xec\x9d\xb4 \xed\x99\x95\xec\x9d\xb8\xeb\x90\x98\xec\xa7\x80 \xec\x95\x8a\xec\x95\x98\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(\n        out / "source_manifest.json",\n        {\n            "original": {\n                split: [asdict(r) for r in retained["original", split]] for split in data.SPLITS\n            },\n            "skin_preparation_pairs": {parent: asdict(row) for parent, row in paired.items()},\n        },\n    )\n    return train, retained["original", "val"], paired\n\n\ndef image_cache(folder, archive_path, train, val, pairs, task, size, profile, *, expanded=None):\n    """Cache exact float32 model inputs on local SSD once for each input resolution."""\n    rows = [*train, *val, *pairs.values()]\n    conditions = ("lighting", "expanded") if expanded is not None else ("lighting",)\n    size_bytes = (len(rows) + len(conditions) * len(val)) * size * size * 3 * 4\n    if shutil.disk_usage(folder).free < size_bytes + 1024**3:\n        raise RuntimeError(\n            f"{task[\'domain\']} \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xea\xb3\xb5\xea\xb0\x84 \xeb\xb6\x80\xec\xa1\xb1 (\xed\x95\x84\xec\x9a\x94 {size_bytes / 1e9:.1f} GB)."\n        )\n    shape = (size, size, 3)\n    cache = np.lib.format.open_memmap(\n        folder / f"inputs_{size}.npy", mode="w+", dtype=np.float32, shape=(len(rows), *shape)\n    )\n    shifted = {\n        name: np.lib.format.open_memmap(\n            folder / f"val_{name}_{size}.npy", mode="w+", dtype=np.float32, shape=(len(val), *shape)\n        )\n        for name in conditions\n    }\n    with zipfile.ZipFile(archive_path) as archive, ThreadPoolExecutor(WORKERS) as pool:\n\n        def prepare(item):\n            i, row = item\n            with tf.device("/CPU:0"):\n                image = exp.image_tensor(archive.read(row.path), size, task["input_policy"])\n            cache[i] = image\n            if row.split == "val":\n                seed = exp.item_seed(row, 1707)\n                shifted["lighting"][i - len(train)] = exp.transform_image(\n                    image, profile, "target_guided", seed\n                )\n                if expanded is not None:\n                    shifted["expanded"][i - len(train)] = camera_extended.transform(\n                        image, expanded, "expanded", seed\n                    )\n\n        for start in range(0, len(rows), 128):\n            list(pool.map(prepare, enumerate(rows[start : start + 128], start)))\n            end = min(start + 128, len(rows))\n            if start == 0 or end % 512 == 0 or end == len(rows):\n                show(f"[{task[\'domain\']} \xc2\xb7 {size}px] \xec\x9e\x85\xeb\xa0\xa5 \xec\xba\x90\xec\x8b\x9c \xec\xa4\x80\xeb\xb9\x84 (\xec\x82\xac\xec\xa7\x84)", end, len(rows))\n    cache.flush()\n    for images in shifted.values():\n        images.flush()\n    lookup = {row.path: i for i, row in enumerate(rows)}\n    paired_indexes = [\n        lookup[pairs[r.path].path] if r.path in pairs else i for i, r in enumerate(train)\n    ]\n    return cache, shifted if expanded is not None else shifted["lighting"], paired_indexes\n\n\ndef pipeline(images, indexes, labels, batch, size, transform=None):\n    def read(index):\n        image = images[int(index)]\n        return image if transform is None else transform(int(index), image)\n\n    def get(index, label):\n        image = tf.numpy_function(read, [index], tf.float32)\n        return tf.ensure_shape(image, (size, size, 3)), label\n\n    ds = tf.data.Dataset.from_tensor_slices(\n        (np.asarray(indexes, np.int32), np.asarray(labels, np.int32))\n    )\n    options = tf.data.Options()\n    options.threading.private_threadpool_size = WORKERS\n    options.threading.max_intra_op_parallelism = 1\n    return ds.with_options(options).map(get, num_parallel_calls=WORKERS).batch(batch).prefetch(2)\n\n\ndef model_for(spec, count, weights):\n    if spec["architecture"] == "pmg":\n        from mediflow_datasets.web_skin_paper_suite import build_pmg_model\n\n        return build_pmg_model(count, spec["size"], weights=weights)\n    size = spec["size"]\n    backbone = keras.applications.EfficientNetB0(\n        include_top=False, weights=weights, input_shape=(size, size, 3)\n    )\n    backbone.trainable = False\n    inputs = keras.Input((size, size, 3))\n    features = keras.layers.GlobalAveragePooling2D()(backbone(inputs, training=False))\n    outputs = keras.layers.Dense(count, activation="softmax")(keras.layers.Dropout(0.3)(features))\n    return keras.Model(inputs, outputs)\n\n\ndef predictor(model, spec):\n    pmg = spec["architecture"] == "pmg"\n    size = spec["size"]\n\n    @tf.function(input_signature=[tf.TensorSpec([None, size, size, 3], tf.float32)])\n    def predict(images):\n        outputs = model(images, training=False)\n        scores = tf.nn.softmax(tf.add_n(outputs), axis=-1) if pmg else outputs\n        return tf.debugging.check_numerics(scores, "validation scores")\n\n    return predict\n\n\ndef steps(model, spec, optimizer, count, *, alignment=None):\n    pmg = spec["architecture"] == "pmg"\n    loss_fn = keras.losses.CategoricalCrossentropy(\n        from_logits=pmg, label_smoothing=spec["label_smoothing"]\n    )\n    variables = model.trainable_variables\n    optimizer.build(variables)\n    if pmg:\n        from mediflow_datasets.web_skin_paper_suite import _jigsaw\n    if alignment is not None:\n        adapter = "sum_logits_softmax" if pmg else "single_softmax"\n        extractor = exp.feature_model(model, adapter)\n        features = keras.Model(model.input, extractor.outputs[-1])\n\n    def update(images, labels, branch=None, factor=1.0, target=None):\n        with tf.GradientTape() as tape:\n            if target is None:\n                outputs = model(images, training=True)\n                logits = outputs[branch] if pmg else outputs\n                mmd = coral = tf.constant(0.0)\n            else:\n                outputs = extractor(images, training=True)\n                logits = outputs[branch if pmg else 0]\n                mmd, coral = domain_alignment.losses(\n                    outputs[-1], features(target, training=False), alignment["reference"]\n                )\n            ce = factor * loss_fn(labels, logits)\n            loss = ce\n            if target is not None:\n                loss = (\n                    loss + alignment["weights"]["mmd"] * mmd + alignment["weights"]["coral"] * coral\n                )\n            loss = tf.debugging.check_numerics(loss, "training loss")\n        grads = tape.gradient(loss, variables)\n        pairs = [\n            (grad, var) for grad, var in zip(grads, variables, strict=True) if grad is not None\n        ]\n        clipped, norm = tf.clip_by_global_norm([g for g, _ in pairs], 1.0)\n        with tf.control_dependencies([tf.debugging.check_numerics(norm, "gradient norm")]):\n            optimizer.apply_gradients(zip(clipped, [v for _, v in pairs], strict=True))\n        return loss, ce, mmd, coral\n\n    signature = [\n        tf.TensorSpec([None, spec["size"], spec["size"], 3], tf.float32),\n        tf.TensorSpec([None], tf.int32),\n    ]\n\n    def update_batch(images, indices, target=None):\n        labels = tf.one_hot(indices, count)\n        if pmg:\n            branch_losses = [\n                update(_jigsaw(images, grid), labels, i)[0] for i, grid in enumerate((8, 4, 2))\n            ]\n            loss, ce, mmd, coral = update(images, labels, 3, 2.0, target)\n            total = tf.add_n(branch_losses)\n            return (total + loss) / 4, (total + ce) / 4, mmd, coral\n        return update(images, labels, target=target)\n\n    @tf.function(input_signature=signature)\n    def train_step(images, indices):\n        return update_batch(images, indices)[0]\n\n    if alignment is not None:\n        aligned_step = tf.function(update_batch, input_signature=[*signature, signature[0]])\n        return train_step, predictor(model, spec), aligned_step\n\n    return train_step, predictor(model, spec)\n\n\ndef epoch_records(folder, epochs):\n    records = sorted(folder.glob("epoch_*.json"))\n    history = [data.read_json(p) for p in records]\n    if len(history) > epochs or [r["epoch"] for r in history] != list(range(1, len(history) + 1)):\n        raise ValueError(f"Incomplete or inconsistent epoch records: {folder}")\n    return history\n\n\ndef save_epoch(checkpoint, folder, local, entry):\n    attempt = "snapshot_" + uuid.uuid4().hex\n    destination = folder / attempt\n    destination.mkdir()\n    local.mkdir(exist_ok=True)\n    local_prefix = checkpoint.write(str(local / "state"))\n    record = {**entry, "snapshot": attempt, "files": {}}\n    for path in sorted(Path(local_prefix).parent.glob("state.*")):\n        shutil.copyfile(path, destination / path.name)\n        record["files"][path.name] = data.sha256(path)\n    if set(record["files"]) != {"state.index", "state.data-00000-of-00001"}:\n        raise RuntimeError("Unexpected checkpoint shard layout")\n    write_progress(folder / f"epoch_{entry[\'epoch\']:03}.json", record)\n    return record\n\n\ndef restore_epoch(checkpoint, folder, record):\n    for name, sha in record["files"].items():\n        if data.sha256(data.local_path(folder, f"{record[\'snapshot\']}/{name}")) != sha:\n            raise ValueError("Epoch checkpoint failed integrity verification")\n    checkpoint.read(str(folder / record["snapshot"] / "state")).assert_consumed()\n\n\ndef publish_model(source, directory, name):\n    temporary = directory / (uuid.uuid4().hex + ".keras.tmp")\n    shutil.copyfile(source, temporary)\n    destination = directory / f"{name}.keras"\n    temporary.replace(destination)\n    return {"file": destination.name, "sha256": data.sha256(source)}\n\n\ndef export_model(model, directory, name, local):\n    local.mkdir(exist_ok=True)\n    scratch = local / f"{name}.keras"\n    model.save(scratch)\n    return publish_model(scratch, directory, name)\n\n\ndef validation_pipelines(cache, shifted, n, val, batch, size):\n    labels = [row.label for row in val]\n    return {\n        "clean": pipeline(cache, range(n, n + len(val)), labels, batch, size),\n        **{\n            name: pipeline(images, range(len(val)), labels, batch, size)\n            for name, images in shifted.items()\n        },\n    }\n\n\ndef validate(predict, datasets, rows, class_count, label):\n    metrics = {}\n    for name, ds in datasets.items():\n        print(f"{label} Validation {name} \xc2\xb7 {len(rows):,}\xec\x9e\xa5", flush=True)\n        scores = np.concatenate([predict(images).numpy() for images, _ in ds])\n        metrics[name] = exp.classification(rows, scores, class_count)\n    return metrics\n\n\ndef phase(model, context, folder, local, spec, phase_name, epochs, rate, *, alignment=None):\n    config, task, train, val, profile, cache, shifted, pairs = context\n    folder.mkdir(exist_ok=True)\n    history = epoch_records(folder, epochs)\n    optimizer = keras.optimizers.Adam(rate)\n    step, predict, *extra_steps = steps(\n        model, spec, optimizer, len(task["classes"]), alignment=alignment\n    )\n    if (phase_name == "aligned") != (alignment is not None):\n        raise ValueError("Alignment inputs must be present exactly for the aligned phase")\n    checkpoint = tf.train.Checkpoint(model=model, optimizer=optimizer)\n    tag = f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']} \xc2\xb7 {phase_name}]"\n    if history:\n        restore_epoch(checkpoint, folder, history[-1])\n        print(f"{tag} Drive\xec\x97\x90\xec\x84\x9c {len(history)}/{epochs} epoch \xeb\xb3\xb5\xec\x9b\x90", flush=True)\n    n, batch, size = len(train), task["batch_size"], spec["size"]\n    expanded = phase_name in ("expanded", "aligned")\n    condition = "expanded" if expanded else "lighting"\n    datasets = validation_pipelines(\n        cache, shifted if expanded else {"lighting": shifted}, n, val, batch, size\n    )\n    for epoch in range(len(history), epochs):\n        started = time.monotonic()\n        keras.utils.set_random_seed(config["seed"] + epoch)\n        order = np.random.default_rng(config["seed"] + epoch).permutation(n)\n        indexes = order\n        if phase_name.startswith("prepare") and spec.get("skin_augmented_preparation"):\n            indexes = [pairs[i] if (i + epoch) % 2 == 0 else i for i in order]\n        transform = None\n        if expanded or phase_name == "lighting":\n            seeds = [exp.item_seed(row, config["seed"], epoch) for row in train]\n            apply = [exp.augmentation_applies(s, config["augmentation_probability"]) for s in seeds]\n\n            def transform(i, image, seeds=seeds, apply=apply):\n                if not apply[i]:\n                    return image\n                if expanded:\n                    return camera_extended.transform(image, profile, "expanded", seeds[i])\n                return exp.transform_image(image, profile, "target_guided", seeds[i])\n\n        label = f"{tag} Epoch {epoch + 1}/{epochs}"\n        batches = math.ceil(n / batch)\n        print(f"{label} \xec\x8b\x9c\xec\x9e\x91 \xc2\xb7 \xed\x95\x99\xec\x8a\xb5 {n:,}\xec\x9e\xa5 \xc2\xb7 {batches} \xeb\xb0\xb0\xec\xb9\x98", flush=True)\n        train_data = pipeline(\n            cache, indexes, [train[i].label for i in order], batch, size, transform\n        )\n        losses = []\n        aligned_metrics = []\n        schedule = {}\n        if alignment is not None:\n            eligible = range(batches - int(n % batch == 1))\n            schedule = domain_alignment.balanced_schedule(\n                alignment["rows"], eligible, config["seed"], epoch\n            )\n            if not schedule:\n                raise ValueError("No Source batch is eligible for feature alignment")\n        for i, (images, labels) in enumerate(train_data, 1):\n            if i - 1 in schedule:\n                target = alignment["images"][schedule[i - 1]]\n                loss, ce, mmd, coral = map(float, extra_steps[0](images, labels, target))\n                losses.append(loss)\n                aligned_metrics.append((ce, mmd, coral))\n            else:\n                losses.append(float(step(images, labels)))\n            if i == 1 or i == batches or i % max(5, batches // 15) == 0:\n                elapsed = time.monotonic() - started\n                remaining = elapsed * (batches / i - 1)\n                show(\n                    label + " \xed\x95\x99\xec\x8a\xb5",\n                    i,\n                    batches,\n                    f"loss={np.mean(losses):.5f} \xc2\xb7 \xeb\x82\xa8\xec\x9d\x80 \xed\x95\x99\xec\x8a\xb5 \xec\x95\xbd {remaining:.0f}\xec\xb4\x88",\n                )\n        train_seconds = time.monotonic() - started\n        metrics = validate(predict, datasets, val, len(task["classes"]), label)\n        validation_seconds = time.monotonic() - started - train_seconds\n        entry = {\n            "epoch": epoch + 1,\n            "train_loss_mean_per_update": float(np.mean(losses)),\n            "training_batches": batches,\n            "optimizer_updates": batches * (4 if spec["architecture"] == "pmg" else 1),\n            "train_seconds": train_seconds,\n            "validation_seconds": validation_seconds,\n            **metrics,\n        }\n        if aligned_metrics:\n            ce, mmd, coral = np.asarray(aligned_metrics).mean(0).tolist()\n            entry["alignment"] = {\n                "updates": len(aligned_metrics),\n                "target_images_used": sum(map(len, schedule.values())),\n                "classification_loss_on_aligned_batches": ce,\n                "mmd_mean": mmd,\n                "coral_mean": coral,\n            }\n            print(\n                f"{label} \xec\xa0\x95\xeb\xa0\xac {len(aligned_metrics)}\xeb\xb0\xb0\xec\xb9\x98 \xc2\xb7 MMD={mmd:.6g} \xc2\xb7 CORAL={coral:.6g}",\n                flush=True,\n            )\n        print(f"{label} Drive \xec\xb2\xb4\xed\x81\xac\xed\x8f\xac\xec\x9d\xb8\xed\x8a\xb8 \xec\xa0\x80\xec\x9e\xa5", flush=True)\n        history.append(save_epoch(checkpoint, folder, local, entry))\n        print(\n            f"{label} \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 loss={entry[\'train_loss_mean_per_update\']:.5f} \xc2\xb7 "\n            f"\xec\x9b\x90\xeb\xb3\xb8 Val F1={metrics[\'clean\'][\'macro_f1\']:.5f} \xc2\xb7 "\n            f"\xec\xa1\xb0\xeb\xaa\x85 Val F1={metrics[\'lighting\'][\'macro_f1\']:.5f}"\n            + (f" \xc2\xb7 \xed\x99\x95\xec\x9e\xa5 Val F1={metrics[\'expanded\'][\'macro_f1\']:.5f}" if expanded else ""),\n            flush=True,\n        )\n    best = max(history, key=lambda r: score(r, expanded or phase_name == "lighting", condition))\n    restore_epoch(checkpoint, folder, best)\n    write_progress(folder / "history.json", history)\n    return best\n\n\ndef score(record, lighting=True, condition="lighting"):\n    clean, shifted = record["clean"]["macro_f1"], record[condition]["macro_f1"]\n    return (min(clean, shifted), clean) if lighting else (clean,)\n\n\ndef trial(context, output, local, spec, job):\n    config, task = context[:2]\n    output.mkdir(exist_ok=True)\n    completed = output / "complete.json"\n    if completed.exists():\n        saved = data.read_json(completed)\n        verify_models(output, saved)\n        print(f"[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9", flush=True)\n        return saved\n    print(f"\\n[{job}/6 \xc2\xb7 {task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\x8b\xa4\xed\x97\x98 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n    keras.backend.clear_session()\n    keras.utils.set_random_seed(config["seed"])\n    count = len(task["classes"])\n    initial_path = output / "initial.json"\n    if initial_path.exists():\n        initial = data.read_json(initial_path)\n        path = data.local_path(output, initial["file"])\n        if data.sha256(path) != initial["sha256"]:\n            raise ValueError("Starting checkpoint changed")\n        model = keras.models.load_model(path, compile=False)\n    else:\n        if spec.get("model"):\n            path = data.local_path(ROOT, spec["model"])\n            if data.sha256(path) != spec["model_sha256"]:\n                raise ValueError("Selected checkpoint changed")\n            model = keras.models.load_model(path, compile=False)\n            initial = publish_model(path, output, "initial")\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xec\xa0\x80\xec\x9e\xa5 \xea\xb0\x80\xec\xa4\x91\xec\xb9\x98\xeb\xa1\x9c \xec\xa1\xb0\xeb\xaa\x85 \xec\xa0\x81\xec\x9d\x91 \xec\x8b\x9c\xec\x9e\x91", flush=True)\n        else:\n            weights = None if (output / "prepare_head/epoch_001.json").exists() else "imagenet"\n            print(f"[{task[\'domain\']} \xc2\xb7 {spec[\'role\']}] \xed\x98\x84\xec\x9e\xac Train\xec\x9c\xbc\xeb\xa1\x9c \xec\xb6\x9c\xeb\xb0\x9c \xeb\xaa\xa8\xeb\x8d\xb8 \xec\xa4\x80\xeb\xb9\x84", flush=True)\n            model = model_for(spec, count, weights)\n            phase(\n                model,\n                context,\n                output / "prepare_head",\n                local,\n                spec,\n                "prepare_head",\n                config["prepare_head_epochs"],\n                config["prepare_head_lr"],\n            )\n            exp.configure_partial(model)\n            phase(\n                model,\n                context,\n                output / "prepare_partial",\n                local,\n                spec,\n                "prepare_partial",\n                config["prepare_partial_epochs"],\n                config["prepare_partial_lr"],\n            )\n            initial = export_model(model, output, "initial", local)\n        write_progress(initial_path, initial)\n    if tuple(model.input_shape[1:]) != (spec["size"], spec["size"], 3):\n        raise ValueError("Initial input shape differs from configuration")\n    if len(model.outputs) != (4 if spec["architecture"] == "pmg" else 1) or any(\n        layer.shape[-1] != count for layer in model.outputs\n    ):\n        raise ValueError("Initial output contract differs from class mapping")\n    exp.configure_partial(model)\n    best = phase(\n        model,\n        context,\n        output / "lighting",\n        local,\n        spec,\n        "lighting",\n        config["epochs"],\n        config["learning_rate"],\n    )\n    saved = {\n        "initial": initial,\n        "best_model": export_model(model, output, "best", local),\n        "model_config": spec,\n        "validation": best,\n        "source_test_evaluated": False,\n        "validation_scope": config.get("validation_scope", "current_source_validation"),\n    }\n    write_progress(completed, saved)\n    return saved\n\n\ndef verify_models(folder, result):\n    for key in ("initial", "best_model"):\n        item = result[key]\n        if data.sha256(data.local_path(folder, item["file"])) != item["sha256"]:\n            raise ValueError(f"Completed checkpoint changed: {folder}/{item[\'file\']}")\n\n\ndef run_domain(domain, source_zip, output, local):\n    config = data.read_json(ROOT / "configs/adaptation_stage1.json")\n    task = config["tasks"][domain]\n    if config.get("initialization_policy") == "saved_only" and any(\n        not spec.get("model") or not spec.get("model_sha256") for spec in task["models"].values()\n    ):\n        raise ValueError("A starting checkpoint is missing from the saved-only bundle")\n    if config.get("validation_scope"):\n        print(\n            f"[{domain}] \xea\xb3\xbc\xea\xb1\xb0 \xed\x95\x99\xec\x8a\xb5 \xec\x9d\xb4\xeb\xa0\xa5\xec\x9d\x98 \xea\xb2\xb9\xec\xb9\xa8 \xec\x97\xac\xeb\xb6\x80 \xeb\xaf\xb8\xed\x99\x95\xec\x9d\xb8 \xc2\xb7 Validation\xec\x9d\x80 \xed\x83\x90\xec\x83\x89 \xeb\xb9\x84\xea\xb5\x90\xec\x9a\xa9", flush=True\n        )\n    reference = data.read_json(ROOT / f"references/{domain}.json")\n    profile = reference["extended_profile"]["lighting"]\n    output.mkdir(parents=True, exist_ok=True)\n    identity = {\n        "config_sha256": object_hash(config),\n        "reference_sha256": object_hash(reference),\n        "code_sha256": data.sha256(Path(__file__)),\n    }\n    marker = output / "identity.json"\n    if marker.exists() and data.read_json(marker) != identity:\n        raise ValueError("Different inputs cannot resume this output")\n    write_progress(marker, identity)\n    write_progress(\n        output / "run_configuration.json",\n        {\n            "identity": identity,\n            "configuration": config,\n            "profile": profile,\n            "tensorflow": tf.__version__,\n            "keras": keras.__version__,\n            "numpy": np.__version__,\n            "gpu": [\n                tf.config.experimental.get_device_details(device)\n                for device in tf.config.list_physical_devices("GPU")\n            ],\n            "code_provenance": data.read_json(ROOT / "bundle_provenance.json"),\n        },\n    )\n    if all((output / role / "complete.json").is_file() for role in ARMS):\n        results = {role: data.read_json(output / role / "complete.json") for role in ARMS}\n        for role, result in results.items():\n            verify_models(output / role, result)\n        print(f"[{domain}] \xeb\x91\x90 \xed\x95\x99\xec\x8a\xb5 \xeb\xaa\xa8\xeb\x91\x90 \xec\x99\x84\xeb\xa3\x8c\xeb\x90\xa8", flush=True)\n        return results\n    train, val, pairs = prepare_source(domain, source_zip, task, reference, output)\n    caches = {}\n    results = {}\n    for i, role in enumerate(ARMS, 1):\n        existing = output / role / "complete.json"\n        if existing.is_file():\n            results[role] = data.read_json(existing)\n            verify_models(output / role, results[role])\n            print(\n                f"[{DOMAINS.index(domain) * 2 + i}/6 \xc2\xb7 {domain} \xc2\xb7 {role}] \xec\x99\x84\xeb\xa3\x8c \xea\xb2\xb0\xea\xb3\xbc \xec\x9e\xac\xec\x82\xac\xec\x9a\xa9",\n                flush=True,\n            )\n            continue\n        spec = {**task["models"][role], "role": role}\n        size = spec["size"]\n        if size not in caches:\n            caches[size] = image_cache(local, source_zip, train, val, pairs, task, size, profile)\n        context = (config, task, train, val, profile, *caches[size])\n        results[role] = trial(\n            context, output / role, local / "saving", spec, DOMAINS.index(domain) * 2 + i\n        )\n    return results\n\n\ndef write_selection(root):\n    selected, table = {}, []\n    for domain in DOMAINS:\n        records = {role: data.read_json(root / domain / role / "complete.json") for role in ARMS}\n        for role, record in records.items():\n            verify_models(root / domain / role, record)\n            table.append(\n                {\n                    "domain": domain,\n                    "role": role,\n                    "best_epoch": record["validation"]["epoch"],\n                    "clean_macro_f1": record["validation"]["clean"]["macro_f1"],\n                    "lighting_macro_f1": record["validation"]["lighting"]["macro_f1"],\n                    "clean_accuracy": record["validation"]["clean"]["accuracy"],\n                    "lighting_accuracy": record["validation"]["lighting"]["accuracy"],\n                    "validation_scope": record.get("validation_scope", "current_source_validation"),\n                }\n            )\n        role = max(ARMS, key=lambda name: score(records[name]["validation"]))\n        record = records[role]\n        selected[domain] = {\n            "role": role,\n            "model_config": record["model_config"],\n            "initial_checkpoint": {\n                "path": f"{domain}/{role}/initial.keras",\n                "sha256": record["initial"]["sha256"],\n            },\n            "stage1_checkpoint": {\n                "path": f"{domain}/{role}/best.keras",\n                "sha256": record["best_model"]["sha256"],\n            },\n        }\n    selection = {\n        "schema": "mediflow_adaptation_stage1_selection_v1",\n        "selected": selected,\n        "criteria": "Maximize worst Val Macro-F1. Break ties by clean Val, then basic.",\n        "source_test_evaluated": False,\n        "target_accuracy": None,\n    }\n    if any(row["validation_scope"].startswith("exploratory_") for row in table):\n        selection["selection_scope"] = "provisional_followup_candidate"\n        selection["historical_train_evaluation_independence_verified"] = False\n    path = root / "selection.json"\n    if path.exists() and data.read_json(path) != selection:\n        raise ValueError("Existing Validation selection differs")\n    with (root / "validation_comparison.csv").open("w", encoding="utf-8-sig", newline="") as f:\n        writer = csv.DictWriter(f, fieldnames=table[0].keys())\n        writer.writeheader()\n        writer.writerows(table)\n    write_progress(path, selection)\n    for row in table:\n        print(row, flush=True)\n    print("1\xec\xb0\xa8 \xec\x99\x84\xeb\xa3\x8c \xc2\xb7 \xeb\xb6\x84\xec\x95\xbc\xeb\xb3\x84 \xec\x84\xa0\xed\x83\x9d:", {d: s["role"] for d, s in selected.items()}, flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--select", type=Path)\n    parser.add_argument("--domain", choices=DOMAINS)\n    parser.add_argument("--source-zip", type=Path)\n    parser.add_argument("--output", type=Path)\n    parser.add_argument("--local", type=Path)\n    args = parser.parse_args()\n    if args.select:\n        write_selection(args.select)\n    elif all((args.domain, args.source_zip, args.output, args.local)):\n        if not tf.config.list_physical_devices("GPU"):\n            raise RuntimeError("TensorFlow GPU\xeb\xa5\xbc \xec\x82\xac\xec\x9a\xa9\xed\x95\xa0 \xec\x88\x98 \xec\x97\x86\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n        run_domain(args.domain, args.source_zip, args.output, args.local)\n    else:\n        parser.error("Provide --select or all domain run parameters")\n\n\nif __name__ == "__main__":\n    main()\n', 'adaptation_stage1_colab.py': b'"""Drive-based stage 1 launcher with live output and scoped interruption."""\n\nfrom __future__ import annotations\n\nimport json\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport uuid\nfrom pathlib import Path\n\nfrom mediflow_datasets import domain_adaptation_batch as batch\nfrom mediflow_datasets.domain_adaptation_io import write_progress\n\nDOMAINS = ("skin", "web_skin", "hair")\n\n\ndef run_live(command, env, log_path, *, input_text=None):\n    with log_path.open("x", encoding="utf-8") as log:\n        process = subprocess.Popen(\n            command,\n            env=env,\n            stdin=subprocess.PIPE if input_text is not None else None,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            encoding="utf-8",\n            errors="replace",\n            bufsize=1,\n        )\n        try:\n            if input_text is not None:\n                with process.stdin:\n                    process.stdin.write(input_text)\n            for line in process.stdout:\n                log.write(line)\n                log.flush()\n                print(line, end="", flush=True)\n            if process.wait():\n                raise subprocess.CalledProcessError(process.returncode, command)\n        finally:\n            process.stdout.close()\n            if process.poll() is None:\n                process.terminate()\n                try:\n                    process.wait(timeout=10)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait()\n\n\ndef copy_source(source, destination):\n    total = source.stat().st_size\n    if shutil.disk_usage(destination.parent).free < total + 1024**3:\n        raise RuntimeError("Colab \xec\x9e\x84\xec\x8b\x9c \xeb\x94\x94\xec\x8a\xa4\xed\x81\xac\xec\x97\x90 \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0 ZIP\xec\x9d\x84 \xeb\xb3\xb5\xec\x82\xac\xed\x95\xa0 \xea\xb3\xb5\xea\xb0\x84\xec\x9d\xb4 \xeb\xb6\x80\xec\xa1\xb1\xed\x95\xa9\xeb\x8b\x88\xeb\x8b\xa4.")\n    with source.open("rb") as reader, destination.open("xb") as writer:\n        done, last = 0, -1\n        while block := reader.read(16 * 1024**2):\n            writer.write(block)\n            done += len(block)\n            progress = min(100, 100 * done // total)\n            if progress >= last + 5 or done == total:\n                print(f"  Drive \xe2\x86\x92 Colab \xeb\xa1\x9c\xec\xbb\xac \xeb\xb3\xb5\xec\x82\xac {progress}% ({done / 1e9:.2f} GB)", flush=True)\n                last = progress\n    if done != total:\n        raise OSError("Drive ZIP \xeb\xb3\xb5\xec\x82\xac\xea\xb0\x80 \xec\xa4\x91\xeb\x8b\xa8\xeb\x90\x90\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n\n\ndef run(runtime, work, project, bundle_sha, my_drive):\n    config = json.loads((runtime / "configs/adaptation_stage1.json").read_text(encoding="utf-8"))\n    python, versions = batch.prepare_python(work)\n    if not versions["gpu"]:\n        raise RuntimeError("Colab \xeb\x9f\xb0\xed\x83\x80\xec\x9e\x84 \xec\x9c\xa0\xed\x98\x95\xec\x97\x90\xec\x84\x9c GPU\xeb\xa5\xbc \xec\x84\xa0\xed\x83\x9d\xed\x95\x9c \xeb\x92\xa4 \xeb\x8b\xa4\xec\x8b\x9c \xec\x8b\xa4\xed\x96\x89\xed\x95\x98\xec\x84\xb8\xec\x9a\x94.")\n    env = batch.python_environment(python, runtime)\n    env.update(PYTHONIOENCODING="utf-8", PYTHONDONTWRITEBYTECODE="1")\n    destination = my_drive / "MediFlow_adaptation_runs" / bundle_sha[:16] / "stage1"\n    destination.mkdir(parents=True, exist_ok=True)\n    marker = destination / "bundle.json"\n    if marker.is_file() and json.loads(marker.read_text(encoding="utf-8"))["sha256"] != bundle_sha:\n        raise RuntimeError("\xec\x9d\xb4 \xea\xb2\xb0\xea\xb3\xbc \xed\x8f\xb4\xeb\x8d\x94\xeb\x8a\x94 \xeb\x8b\xa4\xeb\xa5\xb8 \xeb\xb2\x84\xec\xa0\x84\xec\x9d\x98 \xea\xb3\xb5\xed\x86\xb5 ZIP\xec\x97\x90 \xec\x97\xb0\xea\xb2\xb0\xeb\x8f\xbc \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4.")\n    write_progress(marker, {"sha256": bundle_sha, "stage": 1, "environment": versions})\n    print("1\xec\xb0\xa8 \xea\xb2\xb0\xea\xb3\xbc \xec\xa0\x80\xec\x9e\xa5:", destination, flush=True)\n    failures = {}\n    for domain in DOMAINS:\n        output = destination / domain\n        log = destination / f"{domain}_{uuid.uuid4().hex[:10]}.log"\n        command = [str(python), "-m", "mediflow_datasets.adaptation_stage1"]\n        try:\n            with tempfile.TemporaryDirectory(prefix=f"{domain}_", dir=work) as folder:\n                local = Path(folder)\n                data_zip = local / "source.zip"\n                if not all(\n                    (output / arm / "complete.json").is_file() for arm in ("basic", "selected")\n                ):\n                    print(f"\\n[{domain}] Drive \xeb\x8d\xb0\xec\x9d\xb4\xed\x84\xb0\xec\x85\x8b \xec\xa4\x80\xeb\xb9\x84", flush=True)\n                    source = batch.find_source(project / "datasets", config["tasks"][domain])\n                    copy_source(source, data_zip)\n                run_live(\n                    command\n                    + [\n                        "--domain",\n                        domain,\n                        "--source-zip",\n                        str(data_zip),\n                        "--output",\n                        str(output),\n                        "--local",\n                        str(local),\n                    ],\n                    env,\n                    log,\n                )\n        except Exception as error:\n            failures[domain] = {"error": str(error), "log": log.name if log.is_file() else None}\n            print(f"[{domain}] \xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c: {error}", flush=True)\n    completed = [\n        f"{d}/{a}"\n        for d in DOMAINS\n        for a in ("basic", "selected")\n        if (destination / d / a / "complete.json").is_file() and d not in failures\n    ]\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": False},\n    )\n    if failures:\n        raise RuntimeError(\n            f"\xeb\xaf\xb8\xec\x99\x84\xeb\xa3\x8c \xeb\xb6\x84\xec\x95\xbc\xea\xb0\x80 \xec\x9e\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4. \xec\x99\x84\xeb\xa3\x8c\xeb\x90\x9c \xea\xb8\xb0\xeb\xa1\x9d\xec\x9d\x80 Drive\xec\x97\x90 \xeb\xb3\xb4\xec\xa1\xb4\xed\x96\x88\xec\x8a\xb5\xeb\x8b\x88\xeb\x8b\xa4: {destination}"\n        )\n    run_live(\n        [str(python), "-m", "mediflow_datasets.adaptation_stage1", "--select", str(destination)],\n        env,\n        destination / f"selection_{uuid.uuid4().hex[:10]}.log",\n    )\n    write_progress(\n        destination / "summary.json",\n        {"completed_trials": completed, "failures": failures, "selection_written": True},\n    )\n    print("\xeb\xb9\x84\xea\xb5\x90\xed\x91\x9c:", destination / "validation_comparison.csv", flush=True)\n    return destination\n\n\ndef launch(runtime, work, project, bundle_sha, my_drive):\n    import fcntl\n\n    # Keep two cells in the same Colab VM from touching the same run concurrently.\n    lock = Path("/content") / f"mediflow_stage1_{bundle_sha}.lock"\n    with lock.open("a") as handle:\n        try:\n            fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)\n        except BlockingIOError as error:\n            raise RuntimeError("\xea\xb0\x99\xec\x9d\x80 Colab \xec\x84\xb8\xec\x85\x98\xec\x97\x90\xec\x84\x9c \xec\x9d\xb4\xeb\xaf\xb8 \xec\x8b\xa4\xed\x96\x89 \xec\xa4\x91\xec\x9e\x85\xeb\x8b\x88\xeb\x8b\xa4.") from error\n        return run(runtime, work, project, bundle_sha, my_drive)\n\n\nif __name__ == "__main__":\n    launch(*(Path(arg) if index != 3 else arg for index, arg in enumerate(sys.argv[1:])))\n'}.items():
        (runtime / "src/mediflow_datasets" / name).write_bytes(contents)
    plan_path = work / "final_test_plan.json"
    plan_path.write_bytes(b'{\n  "scope": "source_test_only_historical_checkpoint_independence_unverified",\n  "bundle_sha256": "4ae6e07eb4af122531439519ae2207aad57a1c8dfb8f158152a2093fa65a91fe",\n  "validation_sha256": "5bd16693270fa0962825588a95508412677da866e6d15a745b535bf5d84a32f4",\n  "selection_rule": "Maximize minimum of clean, lighting and expanded Validation Macro-F1; then clean Validation Macro-F1; then earlier arm.",\n  "transform_seed": 1707,\n  "conditions": [\n    "clean",\n    "lighting",\n    "expanded"\n  ],\n  "git_commit": "ffd71eca68ff1e21c145ed56158f4cfa77f7f037",\n  "domains": {\n    "skin": {\n      "start_role": "basic",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/skin/1_training/original/best_model.keras",\n          "sha256": "1b146a6a53742226ec677865aa71bd842a28c9db9c3174b352dd8bc87270770b"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "skin/basic/best.keras",\n          "sha256": "4bca967a479f40745e2d48e81579d442831e955a6193934b5840364fc1e419bb"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "skin/best.keras",\n          "sha256": "9687fa503d8287b880e58e5014cae0a0efd47e750f5ececcc11d813367e7ce6b"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "skin/best.keras",\n          "sha256": "98d9de1af93d31cf2a07cb828d6c29b33b7166f7169da14ca1bc53b7bc2f6f3f"\n        }\n      },\n      "selected_on_validation": "stage1_lighting",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.9965963674000184,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage1_lighting": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage2_expanded": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        },\n        "stage3_aligned": {\n          "clean": 0.9976017479221756,\n          "lighting": 0.9976017479221756,\n          "expanded": 0.9976017479221756\n        }\n      },\n      "reference_sha256": "1837fd554de1363d4695b01190d0625420e5fdde9775313d732fb93e69876172",\n      "manifest_sha256": "36f79b61a42e9ed634ffa7d4c490efefe5e5031a8bbdce1daf630b1e01e7e27d",\n      "test_count": 700\n    },\n    "web_skin": {\n      "start_role": "selected",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/web_skin/selected_models/v2/web_skin_model.keras",\n          "sha256": "83e659dd09a9355135ee0de0197037e81ece962777f59dedae8d9a37b49a3fc4"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "web_skin/selected/best.keras",\n          "sha256": "beb992fa2bf51c28c7b2e760b1c7000bb5b65decef47c0e196ed07c9a62070e5"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "web_skin/best.keras",\n          "sha256": "5e181c1ff064eb1d501c57ccecba91703cfe59b14d02b258152b3ca737c46981"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "web_skin/best.keras",\n          "sha256": "9df65bdf905434e8671afdc034e832170c667193ba803213771d11d5afa5ff71"\n        }\n      },\n      "selected_on_validation": "stage2_expanded",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.8471375958306038,\n          "lighting": 0.8264309294807919,\n          "expanded": 0.8330199818760329\n        },\n        "stage1_lighting": {\n          "clean": 0.8436210626904069,\n          "lighting": 0.8354160598851343,\n          "expanded": 0.8197582784927677\n        },\n        "stage2_expanded": {\n          "clean": 0.8452139236284577,\n          "lighting": 0.8414941635470983,\n          "expanded": 0.8297960345454877\n        },\n        "stage3_aligned": {\n          "clean": 0.8463380823691281,\n          "lighting": 0.8360852895678168,\n          "expanded": 0.8261149876826673\n        }\n      },\n      "reference_sha256": "239245cc7a0804c931ddd45b5b9408474fb8cce5838a00b00d65471b87221b4e",\n      "manifest_sha256": "7c8b41878883193e5dd340ab8a4b7f804a3ab7f37c8159093b663937d997a738",\n      "test_count": 400\n    },\n    "hair": {\n      "start_role": "selected",\n      "models": {\n        "initial": {\n          "stage": 0,\n          "path": "results/hair/selected_models/v2/hair_model.keras",\n          "sha256": "9faa33b8f79f45ef6e7be415473e2ee5225f6195fff04d098828733db23ed3c2"\n        },\n        "stage1_lighting": {\n          "stage": 1,\n          "path": "hair/selected/best.keras",\n          "sha256": "983001b0b116cd913b4cf1e349b47d11ab6246cf78fbaa607dc75b4578108b7f"\n        },\n        "stage2_expanded": {\n          "stage": 2,\n          "path": "hair/best.keras",\n          "sha256": "fa2c59ae6a0b1342b8c693014d0a951c59064a4c8439a313f2520e1ce8c224fd"\n        },\n        "stage3_aligned": {\n          "stage": 3,\n          "path": "hair/best.keras",\n          "sha256": "940b398c7bfdba033df58e99b2639eef8e8eabb0c8c19945dd8f89cfd265e1d7"\n        }\n      },\n      "selected_on_validation": "stage2_expanded",\n      "validation_macro_f1": {\n        "initial": {\n          "clean": 0.7948815320131865,\n          "lighting": 0.7932265946320327,\n          "expanded": 0.7812966127563242\n        },\n        "stage1_lighting": {\n          "clean": 0.7940808319964521,\n          "lighting": 0.790851219358226,\n          "expanded": 0.7717843351060846\n        },\n        "stage2_expanded": {\n          "clean": 0.7933089589962153,\n          "lighting": 0.7891828841548413,\n          "expanded": 0.7892573288809733\n        },\n        "stage3_aligned": {\n          "clean": 0.7924600237364962,\n          "lighting": 0.7891245054928616,\n          "expanded": 0.7884731261645151\n        }\n      },\n      "reference_sha256": "1a44a143fe6f5ef3021865b75f4797ffff16b1e21bae7dc7ca838eb70a09a81f",\n      "manifest_sha256": "c6320fa7d2ef25332d508b3da868d2731d61ce7c8cd3ed2ebc505e6cd0b54717",\n      "test_count": 1252\n    }\n  },\n  "configuration_sha256": "ee9604c652ace8a6d7d0df5656b70ef7512f0ae40f7ef2211f081432b75d9366",\n  "predictor_code_sha256": "6b5aae35bcbe1af85fe19b6a47ebce6ed57bfca780129a82e95dd9dd0c9d4c8a"\n}\n')
    env = {**os.environ, "PYTHONPATH": str(runtime / "src"), "PYTHONIOENCODING": "utf-8",
           "PYTHONUNBUFFERED": "1", "PYTHONDONTWRITEBYTECODE": "1", "MPLBACKEND": "Agg"}
    launch = """from pathlib import Path
import sys
from mediflow_datasets import domain_adaptation_batch as batch
from mediflow_datasets.adaptation_stage1_colab import run_live
import subprocess
work, runtime = map(Path, sys.argv[1:3])
print("Python·TensorFlow 환경과 GPU 확인 중", flush=True)
python, versions = batch.prepare_python(work, executor=run_live)
print("확인된 실행 환경", versions, flush=True)
if not versions["gpu"]:
    raise RuntimeError("Colab에서 GPU 런타임을 선택하고 다시 실행하세요.")
env = batch.python_environment(python, runtime)
check = subprocess.run([str(python), "-c", "import matplotlib"], env=env, capture_output=True)
if check.returncode:
    subprocess.run([str(python), "-m", "pip", "install", "matplotlib>=3.8,<4"], env=env, check=True)
run_live(
    [str(python), "-m", "mediflow_datasets.adaptation_final_test", *sys.argv[3:]],
    env, work / "test.log"
)
"""
    command = [sys.executable, "-u", "-c", launch, str(work), str(runtime),
               "--runtime", str(runtime), "--stages", *map(str, stages),
               "--datasets", str(project / "datasets"), "--plan", str(plan_path),
               "--output", str(out), "--work", str(work)]
    with subprocess.Popen(
        command, env=env, start_new_session=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace", bufsize=1,
    ) as process, log_path.open("x", encoding="utf-8") as log:
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                print(line, end="", flush=True)
            return_code = process.wait()
            if return_code:
                raise RuntimeError(
                    f"최종 Test 실행 실패 · 종료 코드 {return_code} · "
                    f"위의 실제 오류 또는 Drive 로그를 확인하세요. {log_path}"
                )
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
            for diagnostic in work.glob("*.log"):
                shutil.copyfile(diagnostic, log_root / f"{run_id}_{diagnostic.name}")
    local_zip = Path("/content/MediFlow_Final_Test_Results_20261110.zip")
    shutil.make_archive(str(work / "results"), "zip", root_dir=out)
    shutil.copyfile(work / "results.zip", local_zip)
    shutil.copyfile(local_zip, out.with_suffix(".zip"))
print("완료:", out / "test_comparison.csv")
import pandas as pd
from IPython.display import Image, display
from google.colab import files
display(pd.read_csv(out / "test_comparison.csv", dtype=str, keep_default_na=False))
display(Image(filename=str(out / "plots/test_comparison.png")))
display(Image(filename=str(out / "plots/selected_confusion_matrices.png")))
files.download(str(local_zip))
